# Bagian B – Training SigLIP di Google Colab

**Checkpoint:** `google/siglip-so400m-patch14-384`

**Fitur:**
- Mixed precision training (AMP) → hemat VRAM, lebih cepat
- History training disimpan ke JSON di Drive tiap N epoch
- Per-class metrics (precision / recall / F1) di setiap validasi
- Early stopping berdasarkan val loss
- Sanity check visualisasi augmentasi sebelum training
- Ekstraksi embedding → simpan sebagai `.npz`

**Prasyarat:** Sudah menjalankan `preprocessing_lokal.ipynb` dan upload hasilnya ke Google Drive.

---
## CELL 1 – Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('[OK] Google Drive berhasil di-mount.')

---
## CELL 2 – Konfigurasi Utama

In [ ]:
import os

# ============================================================
# KONFIGURASI – sesuaikan ROOT_DIR dengan lokasi upload Anda
# ============================================================
ROOT_DIR    = '/content/drive/MyDrive/Senpro'
DATASET_DIR = os.path.join(ROOT_DIR, 'Dataset')
OUTPUT_DIR  = os.path.join(ROOT_DIR, 'preprocessing_output')

TRAIN_CSV = os.path.join(OUTPUT_DIR, 'train.csv')
VAL_CSV   = os.path.join(OUTPUT_DIR, 'val.csv')
TEST_CSV  = os.path.join(OUTPUT_DIR, 'test.csv')
CW_JSON   = os.path.join(OUTPUT_DIR, 'class_weights.json')

# Model
MODEL_CHECKPOINT = 'google/siglip-so400m-patch14-384'

# Hyperparameter
BATCH_SIZE      = 16        # 16 aman untuk T4 Colab gratis dengan AMP
NUM_EPOCHS      = 20
LEARNING_RATE   = 2e-4
FREEZE_BACKBONE = True      # True = hanya train head (lebih hemat VRAM)
NUM_CLASSES     = 3
DROPOUT_RATE    = 0.3
RANDOM_SEED     = 42

# Mixed precision
USE_AMP = True              # False = matikan AMP jika ada masalah numerik

# Loss function
USE_FOCAL_LOSS = False      # True = Focal Loss; False = CrossEntropy + weights
FOCAL_GAMMA    = 2.0

# Early stopping
EARLY_STOP_PATIENCE = 5     # berhenti jika val_loss tidak turun selama N epoch

# DataLoader
NUM_WORKERS = 2
PIN_MEMORY  = True

# Checkpointing
CHECKPOINT_DIR      = os.path.join(ROOT_DIR, 'checkpoints')
SAVE_EVERY_N_EPOCHS = 2     # simpan checkpoint + history JSON setiap N epoch

# Embedding output
EMBEDDING_DIR = os.path.join(ROOT_DIR, 'embeddings')

# Label mapping
LABEL_TO_IDX = {'Recylable': 0, 'Organik': 1, 'B3': 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EMBEDDING_DIR,  exist_ok=True)

print('=== KONFIGURASI ===')
for name, path in [('ROOT_DIR', ROOT_DIR), ('DATASET_DIR', DATASET_DIR),
                   ('TRAIN_CSV', TRAIN_CSV), ('VAL_CSV', VAL_CSV),
                   ('TEST_CSV', TEST_CSV),   ('CW_JSON', CW_JSON)]:
    status = 'OK' if os.path.exists(path) else 'TIDAK DITEMUKAN'
    print(f'  [{status}] {name}: {path}')

---
## CELL 3 – Cek GPU & Install Dependency

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'[OK] GPU: {gpu_name}')
    print(f'     VRAM: {vram_gb:.1f} GB')
    print(f'     AMP (mixed precision): {"AKTIF" if USE_AMP else "NONAKTIF"}')
else:
    device = torch.device('cpu')
    print('[WARN] GPU tidak tersedia! Training akan sangat lambat.')
    print('  >> Runtime → Change runtime type → GPU → Save')

print(f'\nDevice: {device}')

In [ ]:
!pip install -q --upgrade 'transformers>=4.37.0'
!pip install -q opencv-python-headless pillow tqdm
print('[OK] Instalasi selesai.')

---
## CELL 4 – Import & Reproducibility

In [ ]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
matplotlib.use('Agg')   # aman di Colab
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# AMP (Mixed Precision)
from torch.cuda.amp import autocast, GradScaler

# Per-class metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from transformers import SiglipModel, SiglipImageProcessor

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(RANDOM_SEED)
print(f'[OK] Seed: {RANDOM_SEED} | Semua library berhasil diimport.')

---
## CELL 5 – Muat Manifest & Class Weights

In [ ]:
def load_manifest(csv_path: str, dataset_dir: str) -> pd.DataFrame:
    """
    Baca CSV manifest, bentuk abs_path, verifikasi file ada.
    File yang tidak ditemukan di Drive akan dibuang + dilaporkan.
    """
    df = pd.read_csv(csv_path)
    df['abs_path'] = df['rel_path'].apply(
        lambda p: os.path.join(dataset_dir, p.replace('/', os.sep))
    )
    missing = df[~df['abs_path'].apply(os.path.exists)]
    if len(missing):
        print(f'  [WARN] {len(missing)} file tidak ditemukan → dibuang dari manifest.')
    df = df[df['abs_path'].apply(os.path.exists)].copy().reset_index(drop=True)
    return df


df_train = load_manifest(TRAIN_CSV, DATASET_DIR)
df_val   = load_manifest(VAL_CSV,   DATASET_DIR)
df_test  = load_manifest(TEST_CSV,  DATASET_DIR)

print('=== VERIFIKASI MANIFEST ===')
for name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'\n  {name}: {len(df_s)} sampel')
    for lbl, cnt in sorted(df_s['label'].value_counts().items()):
        print(f'    {lbl}: {cnt}')

In [ ]:
with open(CW_JSON) as f:
    class_weights_dict = json.load(f)

cw_tensor = torch.tensor(
    [class_weights_dict[lbl] for lbl in sorted(LABEL_TO_IDX, key=LABEL_TO_IDX.get)],
    dtype=torch.float32,
).to(device)

print('=== CLASS WEIGHTS ===')
for lbl, w in class_weights_dict.items():
    print(f'  {lbl}: {w:.4f}')
print(f'\n  Tensor (Recylable, Organik, B3): {cw_tensor.tolist()}')

---
## CELL 6 – Augmentasi & Dataset

In [ ]:
def build_aug(split: str, label: str = None) -> transforms.Compose:
    """
    Pipeline augmentasi PIL→PIL.
    Dilakukan SEBELUM SiglipImageProcessor.
    Augmentasi hanya aktif untuk split='train'.
    Kelas B3 mendapat augmentasi lebih agresif.
    """
    if split != 'train':
        return transforms.Compose([
            transforms.Resize((384, 384),
                              interpolation=transforms.InterpolationMode.BICUBIC),
        ])

    if label == 'B3':
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
            transforms.RandomResizedCrop(
                size=(384, 384), scale=(0.7, 1.0),
                interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomGrayscale(p=0.05),
        ])

    # Umum (Recylable, Organik)
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.RandomResizedCrop(
            size=(384, 384), scale=(0.8, 1.0),
            interpolation=transforms.InterpolationMode.BICUBIC),
    ])


class WasteDataset(Dataset):
    """
    Dataset klasifikasi sampah untuk SigLIP.

    Pipeline per item:
    1. Buka gambar PIL (RGB)
    2. Augmentasi (hanya split=train)
    3. SiglipImageProcessor → tensor pixel_values [3, 384, 384], range [-1, 1]
    """

    def __init__(self, df: pd.DataFrame, processor: SiglipImageProcessor,
                 split: str, label_to_idx: dict):
        self.df           = df.reset_index(drop=True)
        self.processor    = processor
        self.split        = split
        self.label_to_idx = label_to_idx

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row   = self.df.iloc[idx]
        label = row['label']

        try:
            image = Image.open(row['abs_path']).convert('RGB')
        except Exception as e:
            warnings.warn(f'[WARN] Gagal buka {row["abs_path"]}: {e}')
            image = Image.new('RGB', (384, 384), color=0)

        # Augmentasi SEBELUM processor
        image = build_aug(self.split, label=label)(image)

        # SiglipImageProcessor: resize + normalize ke [-1, 1]
        pixel_values = self.processor(
            images=image, return_tensors='pt'
        )['pixel_values'].squeeze(0)

        return {
            'pixel_values': pixel_values,
            'labels': torch.tensor(self.label_to_idx[label], dtype=torch.long),
            'path': row['abs_path'],
        }

print('[OK] WasteDataset dan build_aug berhasil didefinisikan.')

---
## CELL 7 – Load Processor & Buat DataLoader

In [ ]:
print(f'[INFO] Memuat SiglipImageProcessor dari: {MODEL_CHECKPOINT}')
processor = SiglipImageProcessor.from_pretrained(MODEL_CHECKPOINT)
print(f'[OK] Image size: {processor.size} | Rescale: {processor.rescale_factor}')
print(f'     Mean: {processor.image_mean} | Std: {processor.image_std}')

# Dataset instances
train_ds = WasteDataset(df_train, processor, 'train', LABEL_TO_IDX)
val_ds   = WasteDataset(df_val,   processor, 'val',   LABEL_TO_IDX)
test_ds  = WasteDataset(df_test,  processor, 'test',  LABEL_TO_IDX)

_kw = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
           persistent_workers=(NUM_WORKERS > 0))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **_kw)

print(f'\n  Train: {len(train_ds)} sampel / {len(train_loader)} batch')
print(f'  Val  : {len(val_ds)} sampel / {len(val_loader)} batch')
print(f'  Test : {len(test_ds)} sampel / {len(test_loader)} batch')

---
## CELL 8 – Sanity Check Visualisasi Augmentasi

Jalankan cell ini **sebelum training** untuk memastikan augmentasi tidak merusak gambar.

In [ ]:
def visualize_augmentations(
    df: pd.DataFrame,
    n_per_class: int = 3,
    save_path: str = None,
):
    """
    Plot n_per_class contoh gambar per kelas:
    - Kolom kiri : gambar ASLI
    - Kolom kanan: gambar setelah augmentasi train

    Disimpan ke save_path jika disediakan.
    Berguna sebagai sanity check sebelum training penuh dimulai.
    """
    classes = sorted(df['label'].unique())
    n_cols  = 2 * len(classes)   # original + aug per kelas
    n_rows  = n_per_class
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 4 * n_rows))
    fig.suptitle('Sanity Check Augmentasi (Kiri: Original | Kanan: Augmented)',
                 fontsize=14, fontweight='bold')

    for col_offset, cls in enumerate(classes):
        subset = df[df['label'] == cls].sample(n=n_per_class,
                                               random_state=RANDOM_SEED)
        aug_transform = build_aug('train', label=cls)

        for row_idx, (_, sample) in enumerate(subset.iterrows()):
            orig = Image.open(sample['abs_path']).convert('RGB')
            aug  = aug_transform(orig)

            ax_orig = axes[row_idx][col_offset * 2]
            ax_aug  = axes[row_idx][col_offset * 2 + 1]

            ax_orig.imshow(orig)
            ax_aug.imshow(aug)

            if row_idx == 0:
                ax_orig.set_title(f'{cls}\nOriginal', fontsize=9)
                ax_aug.set_title(f'{cls}\nAugmented', fontsize=9)

            for ax in (ax_orig, ax_aug):
                ax.axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'[OK] Visualisasi disimpan: {save_path}')

    plt.show()


aug_vis_path = os.path.join(CHECKPOINT_DIR, 'augmentation_sanity_check.png')
visualize_augmentations(df_train, n_per_class=3, save_path=aug_vis_path)

---
## CELL 9 – Model: SigLIP + Classification Head

In [ ]:
class SigLIPClassifier(nn.Module):
    """
    SigLIP vision encoder + lightweight classification head.

    Desain:
    - Backbone  : SiglipModel.vision_model (so400m → embed_dim=1152)
    - Head      : Linear → GELU → Dropout → Linear → num_classes
    - Freeze    : dikontrol oleh freeze_backbone
    """

    def __init__(self, model_checkpoint: str, num_classes: int,
                 dropout: float = 0.3, freeze_backbone: bool = True):
        super().__init__()
        siglip = SiglipModel.from_pretrained(model_checkpoint)
        self.vision_model = siglip.vision_model

        if freeze_backbone:
            for p in self.vision_model.parameters():
                p.requires_grad = False
            print('[INFO] Backbone dibekukan – hanya head yang di-train.')
        else:
            print('[INFO] Full fine-tune – backbone + head.')

        embed_dim  = self.vision_model.config.hidden_size   # 1152
        hidden_dim = embed_dim // 2                          # 576
        print(f'[INFO] Embed dim: {embed_dim}, hidden: {hidden_dim}')

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out    = self.vision_model(pixel_values=pixel_values)
        pooled = out.pooler_output         # [B, 1152]
        return self.classifier(pooled)     # [B, num_classes]

    def encode(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Return raw embedding tanpa classification head (untuk ekstraksi .npz)."""
        with torch.no_grad():
            out = self.vision_model(pixel_values=pixel_values)
        return out.pooler_output           # [B, 1152]


print(f'[INFO] Memuat SigLIP backbone: {MODEL_CHECKPOINT}')
model = SigLIPClassifier(
    model_checkpoint=MODEL_CHECKPOINT,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT_RATE,
    freeze_backbone=FREEZE_BACKBONE,
).to(device)

total_p    = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n  Total params    : {total_p:,}')
print(f'  Trainable params: {trainable_p:,} ({trainable_p/total_p*100:.1f}%)')

---
## CELL 10 – Loss, Optimizer, Scheduler, GradScaler

In [ ]:
class FocalLoss(nn.Module):
    """FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt). Alternatif CrossEntropy."""

    def __init__(self, weight: torch.Tensor = None, gamma: float = 2.0):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


criterion = (
    FocalLoss(weight=cw_tensor, gamma=FOCAL_GAMMA)
    if USE_FOCAL_LOSS
    else nn.CrossEntropyLoss(weight=cw_tensor)
)
print(f'[OK] Loss: {"FocalLoss" if USE_FOCAL_LOSS else "CrossEntropyLoss"} + class weights')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6,
)

# GradScaler untuk AMP – otomatis di-skip jika USE_AMP=False
# enabled=False → GradScaler jadi no-op, tidak perlu kode cabang
scaler = GradScaler(enabled=USE_AMP)

print(f'[OK] Optimizer: AdamW (lr={LEARNING_RATE})')
print(f'[OK] Scheduler: CosineAnnealingLR')
print(f'[OK] GradScaler (AMP): enabled={USE_AMP}')

---
## CELL 11 – Helper: Train / Eval / Metrics / Checkpoint / History

In [ ]:
# ── Training satu epoch (dengan AMP) ─────────────────────────
def train_one_epoch(model, loader, optimizer, criterion,
                    scaler, device) -> tuple:
    """Return (avg_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct    = 0
    n          = 0

    for batch in tqdm(loader, desc='  Train', leave=False, unit='batch'):
        pv     = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        optimizer.zero_grad()

        # autocast: forward pass di float16
        with autocast(enabled=USE_AMP):
            logits = model(pv)
            loss   = criterion(logits, labels)

        # scaler: scale loss → backward → unscale → clip → step
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(-1) == labels).sum().item()
        n          += labels.size(0)

    return total_loss / n, correct / n


# ── Evaluasi (loss + accuracy + per-class metrics) ────────────
@torch.no_grad()
def evaluate(model, loader, criterion, device) -> dict:
    """
    Return dict dengan:
      loss, accuracy,
      precision_{label}, recall_{label}, f1_{label} untuk setiap kelas.
    Kunci B3 recall: 'recall_B3' – dipantau tiap epoch.
    """
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for batch in tqdm(loader, desc='  Eval', leave=False, unit='batch'):
        pv     = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(pv)
            loss   = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    n        = len(all_labels)
    avg_loss = total_loss / n
    accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / n

    # Per-class precision / recall / F1
    label_names = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )

    metrics = {'loss': avg_loss, 'accuracy': accuracy}
    for i, lname in enumerate(label_names):
        metrics[f'precision_{lname}'] = round(float(precision[i]), 4)
        metrics[f'recall_{lname}']    = round(float(recall[i]),    4)
        metrics[f'f1_{lname}']        = round(float(f1[i]),        4)

    return metrics


# ── Checkpoint ────────────────────────────────────────────────
def save_checkpoint(model, optimizer, scheduler, scaler,
                    epoch: int, val_loss: float,
                    checkpoint_dir: str) -> str:
    path = os.path.join(checkpoint_dir, f'checkpoint_epoch{epoch:03d}.pt')
    torch.save({
        'epoch': epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'val_loss': val_loss,
    }, path)
    print(f'  [Checkpoint] {path}')
    return path


def load_checkpoint(model, optimizer, scheduler, scaler,
                    path: str) -> int:
    """Muat checkpoint. Return epoch terakhir."""
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    print(f'[OK] Resume dari epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f}')
    return ckpt['epoch']


# ── History JSON (simpan ke Drive) ────────────────────────────
HISTORY_PATH = os.path.join(CHECKPOINT_DIR, 'training_history.json')

def save_history(history: dict) -> None:
    """Tulis history dict ke JSON di Drive. Aman untuk di-call tiap epoch."""
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f, indent=2)


def load_history() -> dict:
    """Muat history yang sudah ada (untuk resume setelah disconnect)."""
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            return json.load(f)
    return {}


print('[OK] Semua helper function berhasil didefinisikan.')

---
## CELL 12 – Training Loop (AMP + Early Stopping + History JSON)

In [ ]:
# ── Resume dari checkpoint jika ada ──────────────────────────
# Ganti nilai RESUME_CHECKPOINT untuk melanjutkan setelah disconnect:
#   RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch010.pt')
RESUME_CHECKPOINT = None

start_epoch = 0
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    start_epoch = load_checkpoint(model, optimizer, scheduler, scaler,
                                  RESUME_CHECKPOINT)

# Muat history yang sudah ada (jika ada)
history = load_history()
if not history:
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   [],
        # Per-class metrics per epoch
        'val_recall_B3':        [],
        'val_precision_B3':     [],
        'val_f1_B3':            [],
        'val_recall_Recylable': [],
        'val_recall_Organik':   [],
    }
    print('[INFO] History baru dimulai.')
else:
    print(f'[INFO] History lama dimuat ({len(history["train_loss"])} epoch).')

# ── Early stopping state ──────────────────────────────────────
best_val_loss    = float('inf')
no_improve_count = 0
best_model_path  = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

# Jika resume, ambil best_val_loss dari history yang ada
if history['val_loss']:
    best_val_loss = min(history['val_loss'])

print(f'\nTraining: max {NUM_EPOCHS} epoch | batch={BATCH_SIZE}')
print(f'AMP={USE_AMP} | freeze={FREEZE_BACKBONE} | early_stop_patience={EARLY_STOP_PATIENCE}')
print(f'Device: {device}')
print('=' * 65)

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')

    # Train
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device
    )

    # Validasi + per-class metrics
    val_metrics = evaluate(model, val_loader, criterion, device)
    v_loss = val_metrics['loss']
    v_acc  = val_metrics['accuracy']

    scheduler.step()

    # Simpan ke history
    history['train_loss'].append(round(tr_loss, 6))
    history['train_acc'].append(round(tr_acc, 6))
    history['val_loss'].append(round(v_loss, 6))
    history['val_acc'].append(round(v_acc, 6))
    history['val_recall_B3'].append(val_metrics['recall_B3'])
    history['val_precision_B3'].append(val_metrics['precision_B3'])
    history['val_f1_B3'].append(val_metrics['f1_B3'])
    history['val_recall_Recylable'].append(val_metrics['recall_Recylable'])
    history['val_recall_Organik'].append(val_metrics['recall_Organik'])

    # Print ringkasan epoch
    print(f'  Train  → loss: {tr_loss:.4f} | acc: {tr_acc*100:.2f}%')
    print(f'  Val    → loss: {v_loss:.4f} | acc: {v_acc*100:.2f}%')
    print(f'  Recall → Recylable: {val_metrics["recall_Recylable"]:.4f} | '
          f'Organik: {val_metrics["recall_Organik"]:.4f} | '
          f'B3: {val_metrics["recall_B3"]:.4f}   ← pantau B3!')
    print(f'  LR     : {scheduler.get_last_lr()[0]:.2e}')

    # Simpan history JSON ke Drive setiap N epoch
    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        save_history(history)
        save_checkpoint(model, optimizer, scheduler, scaler,
                        epoch + 1, v_loss, CHECKPOINT_DIR)

    # Best model
    if v_loss < best_val_loss:
        best_val_loss    = v_loss
        no_improve_count = 0
        torch.save(model.state_dict(), best_model_path)
        save_history(history)   # simpan history saat best model update
        print(f'  ★ Best model diperbarui (val_loss={v_loss:.4f})')
    else:
        no_improve_count += 1
        print(f'  No improvement: {no_improve_count}/{EARLY_STOP_PATIENCE}')
        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(f'\n[EARLY STOP] val_loss tidak turun selama {EARLY_STOP_PATIENCE} epoch.'
                  f' Training dihentikan di epoch {epoch+1}.')
            break

# Simpan history final
save_history(history)
print(f'\n[OK] Training selesai. Best val_loss: {best_val_loss:.4f}')
print(f'[OK] History disimpan di: {HISTORY_PATH}')

---
## CELL 13 – Plot Kurva Training (dari History JSON)

In [ ]:
# Cell ini bisa dijalankan kapan saja, bahkan setelah sesi terputus,
# karena membaca dari file JSON di Drive – bukan variabel di memori.
history = load_history()
epochs_done = len(history['train_loss'])
x = range(1, epochs_done + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Loss
axes[0].plot(x, history['train_loss'], label='Train Loss', marker='o', markersize=3)
axes[0].plot(x, history['val_loss'],   label='Val Loss',   marker='s', markersize=3)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Accuracy
axes[1].plot(x, [a*100 for a in history['train_acc']], label='Train Acc', marker='o', markersize=3)
axes[1].plot(x, [a*100 for a in history['val_acc']],   label='Val Acc',   marker='s', markersize=3)
axes[1].set_title('Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Per-class Recall
axes[2].plot(x, history['val_recall_B3'],        label='B3 Recall',        marker='^', markersize=4, color='red')
axes[2].plot(x, history['val_recall_Recylable'], label='Recylable Recall',  marker='o', markersize=3, color='blue')
axes[2].plot(x, history['val_recall_Organik'],   label='Organik Recall',    marker='s', markersize=3, color='green')
axes[2].set_title('Val Recall per Kelas')
axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1.05)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()

curve_path = os.path.join(CHECKPOINT_DIR, 'training_curves.png')
plt.savefig(curve_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'[OK] Kurva disimpan: {curve_path}')

---
## CELL 14 – Evaluasi Final di Test Set

In [ ]:
# Muat best model
model.load_state_dict(torch.load(best_model_path, map_location=device))
print(f'[OK] Best model dimuat dari: {best_model_path}')

test_metrics = evaluate(model, test_loader, criterion, device)

print('\n=== EVALUASI TEST SET ===')
print(f'  Loss    : {test_metrics["loss"]:.4f}')
print(f'  Accuracy: {test_metrics["accuracy"]*100:.2f}%')
print('\n  Per kelas:')
label_names = [IDX_TO_LABEL[i] for i in range(NUM_CLASSES)]
for lname in label_names:
    print(f'    {lname:10s} | P={test_metrics[f"precision_{lname}"]:.4f} '
          f'| R={test_metrics[f"recall_{lname}"]:.4f} '
          f'| F1={test_metrics[f"f1_{lname}"]:.4f}')

In [ ]:
# Classification report lengkap + confusion matrix
all_preds, all_lbls = [], []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Prediksi test', unit='batch'):
        pv     = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        with autocast(enabled=USE_AMP):
            logits = model(pv)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_lbls.extend(labels.cpu().tolist())

print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(all_lbls, all_preds, target_names=label_names))

print('=== CONFUSION MATRIX ===')
cm = confusion_matrix(all_lbls, all_preds)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print(cm_df)

---
## CELL 15 – Ekstraksi Embedding → Simpan `.npz`

Embedding diekstrak dari **pooler_output** vision encoder (1152-dim untuk so400m).  
Disimpan per split sebagai `embeddings_{split}.npz` di `EMBEDDING_DIR`.  
File `.npz` memiliki array:
- `embeddings` – shape `[N, 1152]`
- `labels`     – shape `[N]`, integer kelas
- `label_names`– shape `[N]`, string nama kelas
- `paths`      – shape `[N]`, path absolut gambar

In [ ]:
@torch.no_grad()
def extract_embeddings(
    model: SigLIPClassifier,
    loader: DataLoader,
    device: torch.device,
    split_name: str,
    output_dir: str,
) -> str:
    """
    Ekstrak embedding dari vision encoder untuk seluruh dataset dalam loader.
    Simpan sebagai .npz di output_dir.

    Menggunakan model.encode() (pooler_output tanpa classification head).
    AMP dipakai agar konsisten dengan training.

    Returns:
        path ke file .npz yang disimpan.
    """
    model.eval()

    all_embs    = []
    all_labels  = []
    all_lnames  = []
    all_paths   = []

    for batch in tqdm(loader, desc=f'Ekstrak [{split_name}]', unit='batch'):
        pv     = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels']
        paths  = batch['path']

        with autocast(enabled=USE_AMP):
            # encode() → pooler_output, fp16 jika AMP aktif
            emb = model.encode(pv)   # [B, 1152]

        all_embs.append(emb.float().cpu().numpy())   # simpan sebagai float32
        all_labels.extend(labels.tolist())
        all_lnames.extend([IDX_TO_LABEL[l] for l in labels.tolist()])
        all_paths.extend(list(paths))

    embeddings  = np.concatenate(all_embs, axis=0)  # [N, 1152]
    label_arr   = np.array(all_labels,  dtype=np.int32)
    lname_arr   = np.array(all_lnames,  dtype=object)
    path_arr    = np.array(all_paths,   dtype=object)

    out_path = os.path.join(output_dir, f'embeddings_{split_name}.npz')
    np.savez_compressed(
        out_path,
        embeddings  = embeddings,
        labels      = label_arr,
        label_names = lname_arr,
        paths       = path_arr,
    )

    print(f'[OK] embeddings_{split_name}.npz → shape={embeddings.shape}')
    print(f'     Disimpan: {out_path}')
    return out_path


print('[OK] extract_embeddings berhasil didefinisikan.')

In [ ]:
# Jalankan ekstraksi untuk semua split menggunakan best model
# Pastikan best model sudah dimuat (Cell 14 sudah memuatnya)

print('=== EKSTRAKSI EMBEDDING ===')
print(f'Output dir: {EMBEDDING_DIR}')
print(f'Menggunakan model: best_model.pt\n')

for split_name, loader in [
    ('train', train_loader),
    ('val',   val_loader),
    ('test',  test_loader),
]:
    extract_embeddings(model, loader, device, split_name, EMBEDDING_DIR)

print('\n[OK] Semua embedding berhasil diekstrak dan disimpan.')

---
## CELL 16 – Verifikasi & Contoh Load Embedding `.npz`

Cell ini menunjukkan cara membaca `.npz` di kemudian hari (misal untuk clustering, UMAP, dll).

In [ ]:
# Load dan verifikasi salah satu file .npz
npz_path = os.path.join(EMBEDDING_DIR, 'embeddings_test.npz')
data = np.load(npz_path, allow_pickle=True)

print('=== VERIFIKASI embeddings_test.npz ===')
print(f'  Keys         : {list(data.keys())}')
print(f'  embeddings   : {data["embeddings"].shape}, dtype={data["embeddings"].dtype}')
print(f'  labels       : {data["labels"].shape}')
print(f'  label_names  : contoh = {data["label_names"][:5].tolist()}')
print(f'  paths        : contoh = {[Path(p).name for p in data["paths"][:3].tolist()]}')

# Distribusi kelas dalam file test
from collections import Counter
dist = Counter(data['label_names'].tolist())
print(f'\n  Distribusi kelas:')
for lbl, cnt in sorted(dist.items()):
    print(f'    {lbl}: {cnt}')

print('\n[OK] File .npz siap digunakan untuk downstream task (UMAP, clustering, dll).')